# CiteScope — Modelado · Fase 4a: Tuning de hiperparámetros

**Objetivo.** Buscar mejores hiperparámetros para el pipeline ganador (TF-IDF + Logistic Regression sobre `text_enriched`) **sin sobreajustar a validación**, usando validación cruzada por grupos sobre `train`.

**Entregables de esta fase.**
- Búsqueda con **`GridSearchCV` + `StratifiedGroupKFold`** (agrupando por `citing_arxiv_id`), optimizando **Macro F1**.
- Mejor configuración elegida por CV y confirmación en el conjunto de **validación**.
- Comparación contra el enriquecido por defecto (Macro F1 = 0,628) y registro en `models/artifacts/tuning_results.csv`.

> Por qué CV por grupos: `val` es un único split de 800 filas; elegir hiperparámetros mirándolo directamente arriesga sobreajustar a ese split. La CV sobre `train` (respetando la agrupación anti-fuga) da una estimación robusta y deja `val` como verificación limpia. El **test sigue intacto** hasta la evaluación final.


## Carga de datos, split y grupos

Cargamos el dataset, construimos `text_enriched`, adjuntamos el split de la Fase 1 y derivamos los grupos (`citing_arxiv_id`) del subconjunto `train` para la validación cruzada.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
TARGET = "citing_primary_category"
BASELINE_ENRICHED_VAL = 0.6277  # Macro F1 del enriquecido por defecto (Fase 3)

REPO_ROOT = Path.cwd().parent
DATASET_DIR = REPO_ROOT / "Dataset"
ARTIFACTS_DIR = Path.cwd() / "artifacts"

CANONICAL = DATASET_DIR / "unarxive_microproyecto.jsonl"
LOCAL_COPY = DATASET_DIR / "copy_unarxive_microproyecto.jsonl"
DATA_PATH = CANONICAL if CANONICAL.exists() else LOCAL_COPY

df = pd.read_json(DATA_PATH, lines=True, dtype={"citing_arxiv_id": "string"})
ctx = df["citation_context"].fillna("").astype(str).str.strip()
title = df["cited_title"].fillna("").astype(str).str.strip()
abstract = df["cited_abstract"].fillna("").astype(str).str.strip()
df["text_enriched"] = ["\n".join(p for p in (c, t, a) if p) for c, t, a in zip(ctx, title, abstract)]

split_map = pd.read_csv(ARTIFACTS_DIR / "split_assignment.csv")[["citation_id", "split"]]
df = df.merge(split_map, on="citation_id", how="left")
df["_group"] = df["citing_arxiv_id"].where(df["citing_arxiv_id"].notna(), df["citation_id"].astype("string"))

train = df[df["split"] == "train"]
val = df[df["split"] == "val"]
groups_train = train["_group"]

print(f"train: {len(train)} | val: {len(val)} | grupos en train: {groups_train.nunique()}")


train: 2400 | val: 800 | grupos en train: 915


## Búsqueda de hiperparámetros

Grilla sobre TF-IDF (`ngram_range`, `min_df`, `max_features`) y LogReg (`C`, `class_weight`), con CV de 5 folds `StratifiedGroupKFold` sobre `train`, `scoring="f1_macro"` y `n_jobs=-1`.


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ("tfidf", TfidfVectorizer(sublinear_tf=True)),
    ("clf", LogisticRegression(max_iter=1000, random_state=SEED)),
])

param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [2, 3, 5],
    "tfidf__max_features": [None, 50000],
    "clf__C": [0.3, 1, 3, 10],
    "clf__class_weight": [None, "balanced"],
}

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
search = GridSearchCV(pipe, param_grid, scoring="f1_macro", cv=cv, n_jobs=-1, verbose=1)
search.fit(train["text_enriched"], train[TARGET], groups=groups_train)

print(f"Mejor Macro F1 (CV): {search.best_score_:.4f}")
print("Mejores hiperparámetros:")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")


Fitting 5 folds for each of 96 candidates, totalling 480 fits


/Users/germanrodriguez/Desktop/Code/CSCO/MAIA4401_MicroProyecto/.venv/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Mejor Macro F1 (CV): 0.6183
Mejores hiperparámetros:
  clf__C: 1
  clf__class_weight: None
  tfidf__max_features: None
  tfidf__min_df: 5
  tfidf__ngram_range: (1, 1)


## Top configuraciones (CV)

Revisamos las mejores combinaciones según la CV para entender qué hiperparámetros importan y qué tan estable es la elección.


In [3]:
cv_results = pd.DataFrame(search.cv_results_)
cols = [
    "param_tfidf__ngram_range", "param_tfidf__min_df", "param_tfidf__max_features",
    "param_clf__C", "param_clf__class_weight", "mean_test_score", "std_test_score",
]
top = cv_results[cols].sort_values("mean_test_score", ascending=False).head(10).reset_index(drop=True)
top["mean_test_score"] = top["mean_test_score"].round(4)
top["std_test_score"] = top["std_test_score"].round(4)
print("Top 10 configuraciones por Macro F1 (CV):")
print(top.to_string(index=False))


Top 10 configuraciones por Macro F1 (CV):
param_tfidf__ngram_range  param_tfidf__min_df param_tfidf__max_features  param_clf__C param_clf__class_weight  mean_test_score  std_test_score
                  (1, 1)                    5                     50000           1.0                balanced           0.6183          0.0147
                  (1, 1)                    5                      None           1.0                     NaN           0.6183          0.0147
                  (1, 1)                    5                     50000           1.0                     NaN           0.6183          0.0147
                  (1, 1)                    5                      None           1.0                balanced           0.6183          0.0147
                  (1, 1)                    3                     50000           1.0                balanced           0.6181          0.0193
                  (1, 1)                    3                     50000           1.0               

## Confirmación en validación

Tomamos el mejor estimador (reajustado por `GridSearchCV` sobre todo `train`) y lo evaluamos en `val`, comparándolo con el enriquecido por defecto (0,628) para ver si el tuning aporta una mejora real fuera de la CV.


In [4]:
from sklearn.metrics import accuracy_score, classification_report, f1_score

best_model = search.best_estimator_
val_pred = best_model.predict(val["text_enriched"])
macro_f1_val = f1_score(val[TARGET], val_pred, average="macro")
acc_val = accuracy_score(val[TARGET], val_pred)

print(f"Enriquecido por defecto (val):  Macro F1 = {BASELINE_ENRICHED_VAL:.4f}")
print(f"Tuneado (val):                  Macro F1 = {macro_f1_val:.4f}  (delta {macro_f1_val - BASELINE_ENRICHED_VAL:+.4f})")
print(f"Tuneado (val):                  Accuracy = {acc_val:.4f}")
print("\nReporte por clase (tuneado, val):\n")
print(classification_report(val[TARGET], val_pred, digits=3))


Enriquecido por defecto (val):  Macro F1 = 0.6277
Tuneado (val):                  Macro F1 = 0.6145  (delta -0.0132)
Tuneado (val):                  Accuracy = 0.6150

Reporte por clase (tuneado, val):

              precision    recall  f1-score   support

       cs.AI      0.491     0.530     0.510       100
       cs.CL      0.674     0.620     0.646       100
       cs.CV      0.642     0.770     0.700       100
       cs.IR      0.629     0.660     0.644       100
       cs.LG      0.355     0.330     0.342       100
       cs.MA      0.677     0.670     0.673       100
       cs.NE      0.680     0.660     0.670       100
       cs.RO      0.791     0.680     0.731       100

    accuracy                          0.615       800
   macro avg      0.617     0.615     0.614       800
weighted avg      0.617     0.615     0.614       800



## Registro de resultados

Guardamos la mejor configuración y su desempeño en `models/artifacts/tuning_results.csv` para la comparación final y para re-instrumentar en MLflow.


In [5]:
ARTIFACTS_DIR.mkdir(exist_ok=True)

tuning_row = {
    "modelo": "logreg", "input": "text_enriched",
    "cv_macro_f1": round(search.best_score_, 4),
    "macro_f1_val": round(macro_f1_val, 4),
    "accuracy_val": round(acc_val, 4),
    "delta_vs_default": round(macro_f1_val - BASELINE_ENRICHED_VAL, 4),
    "seed": SEED,
    **{k: str(v) for k, v in search.best_params_.items()},
}
tuning_df = pd.DataFrame([tuning_row])
tuning_df.to_csv(ARTIFACTS_DIR / "tuning_results.csv", index=False)

print("Guardado en:", (ARTIFACTS_DIR / "tuning_results.csv").relative_to(REPO_ROOT))
tuning_df


Guardado en: models/artifacts/tuning_results.csv


,modelo,input,cv_macro_f1,macro_f1_val,accuracy_val,delta_vs_default,seed,clf__C,clf__class_weight,tfidf__max_features,tfidf__min_df,tfidf__ngram_range
0,logreg,text_enriched,0.6183,0.6145,0.615,-0.0132,42,1,None,None,5,"(1, 1)"
